# Neural Network Hyperparameter Sweeps - Weighted Season Average

This notebook runs W&B sweeps to find optimal hyperparameters for neural network models.

In [ ]:
# Only required on GPU Hub
%pip install dotenv wandb xgboost catboost lightning

In [1]:
import sys

sys.path.append("..")

import dotenv
import wandb

from src.api.run.neural_network import sweep_neural_network
from src.api.sweep import wandb_sweep
from src.models.neural_network.submission import create_submission

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Sweep 1: MSE Loss

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Weighted Average)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                        [512, 1024, 512, 256, 128],
                        [512, 1024, 512, 256, 128, 64],
                        [512, 1024, 512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "mse"},
                "scheduler": {"values": [None, "step", "cosine", "exponential", "reduce_on_plateau"]},
                "scheduler_step_size": {"values": [10, 15, 20, 25, 30]},
                "scheduler_gamma": {"distribution": "log_uniform_values", "min": 0.7, "max": 0.99},
                "scheduler_patience": {"values": [3, 5, 7]},
                "num_workers": {"value": 16},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 0},
                "start_season": {"value": 0},
                "num_features": {"distribution": "int_uniform", "min": 10, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model (Loss: MSE)

In [3]:
create_submission(season=2025, artifact_name="mrc96iou:v2", submission_affix="weighted_avg")

Config: {'config': "NeuralNetworkHyperparamConfig(hidden_layers=[64, 32], activation='gelu', dropout=0.3591515261148852, batch_norm=True, input_dropout=0.0, learning_rate=0.002633551270836464, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.03873843391709941, momentum=0.9, scheduler='exponential', scheduler_step_size=25, scheduler_gamma=0.9477039164797748, scheduler_patience=3, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='mse', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)", 'input_dim': 85, 'run_config': {'data_loader': 'weighted_season_average', 'num_features': 89, 'start_season': 0, 'valid_season': 0, 'data_loader_config': {'regular_weight': 0.9943383083708116, 'tourney_weight': 0.9826034734482154, 'discount_factor': 0.9539002462198868}}, 'neural_network_config': {'seed': 42, 'dropout': 0.3591515261148852, 'optimizer': 'adamw', 'sche

Seed set to 42
wandb:   1 of 1 files downloaded.  


Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_neural_network_weighted_avg_2025.csv


## Sweep 2: BCE Loss

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Weighted Average, BCE)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                        [512, 1024, 512, 256, 128],
                        [512, 1024, 512, 256, 128, 64],
                        [512, 1024, 512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 5},
                "seed": {"value": 42},
                "loss_function": {"value": "bce"},
                "scheduler": {"values": [None, "step", "cosine", "exponential", "reduce_on_plateau"]},
                "scheduler_step_size": {"values": [10, 15, 20, 25, 30]},
                "scheduler_gamma": {"distribution": "log_uniform_values", "min": 0.7, "max": 0.99},
                "scheduler_patience": {"values": [3, 5, 7]},
                "num_workers": {"value": 16},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 0},
                "start_season": {"value": 0},
                "num_features": {"distribution": "int_uniform", "min": 10, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model (Loss: BCE)

In [4]:
create_submission(season=2025, artifact_name="uhphyp9g:v0", submission_affix="weighted_avg_bce")

Config: {'config': "NeuralNetworkHyperparamConfig(hidden_layers=[512, 1024, 512, 256, 128], activation='elu', dropout=0.33204003666167997, batch_norm=True, input_dropout=0.0, learning_rate=0.0001930682425271947, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.0140726555574751, momentum=0.9, scheduler='cosine', scheduler_step_size=15, scheduler_gamma=0.7461611608491112, scheduler_patience=7, early_stopping=True, early_stopping_patience=5, early_stopping_min_delta=0.0001, loss_function='bce', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)", 'input_dim': 85, 'run_config': {'data_loader': 'weighted_season_average', 'num_features': 100, 'start_season': 0, 'valid_season': 0, 'data_loader_config': {'regular_weight': 0.9500550803809764, 'tourney_weight': 0.8012747680933215, 'discount_factor': 0.9892383336280371}}, 'neural_network_config': {'seed': 42, 'dropout': 0.33204003666167997, 'optimizer': 

Seed set to 42
wandb:   1 of 1 files downloaded.  


Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_neural_network_weighted_avg_bce_2025.csv


## Sweep 3: Default Features (num_features=0)

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Weighted Average, Default Features)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                        [512, 1024, 512, 256, 128],
                        [512, 1024, 512, 256, 128, 64],
                        [512, 1024, 512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "mse"},
                "scheduler": {"values": [None, "step", "cosine", "exponential", "reduce_on_plateau"]},
                "scheduler_step_size": {"values": [10, 15, 20, 25, 30]},
                "scheduler_gamma": {"distribution": "log_uniform_values", "min": 0.7, "max": 0.99},
                "scheduler_patience": {"values": [3, 5, 7]},
                "num_workers": {"value": 16},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 0},
                "start_season": {"value": 0},
                "num_features": {"value": 0},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model (Default Features)

In [5]:
create_submission(season=2025, artifact_name="c06n3ys3:v1", submission_affix="weighted_avg_default_features")

Config: {'config': "NeuralNetworkHyperparamConfig(hidden_layers=[512, 1024, 512, 256, 128], activation='gelu', dropout=0.44674465666210056, batch_norm=True, input_dropout=0.0, learning_rate=0.0006474500741554385, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.0356647221248357, momentum=0.9, scheduler='reduce_on_plateau', scheduler_step_size=20, scheduler_gamma=0.8792257585761334, scheduler_patience=5, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='mse', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)", 'input_dim': 29, 'run_config': {'data_loader': 'weighted_season_average', 'num_features': 0, 'start_season': 0, 'valid_season': 0, 'data_loader_config': {'regular_weight': 0.5822757415769353, 'tourney_weight': 0.8294787599573641, 'discount_factor': 0.9741346463137044}}, 'neural_network_config': {'seed': 42, 'dropout': 0.44674465666210056, 'o

Seed set to 42
wandb:   1 of 1 files downloaded.  


Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_neural_network_weighted_avg_default_features_2025.csv


## Sweep 4: BCE with Entropy Loss

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Weighted Average, BCEwEntropy)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                        [512, 1024, 512, 256, 128],
                        [512, 1024, 512, 256, 128, 64],
                        [512, 1024, 512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "bce_entropy"},
                "scheduler": {"values": [None, "step", "cosine", "exponential", "reduce_on_plateau"]},
                "scheduler_step_size": {"values": [10, 15, 20, 25, 30]},
                "scheduler_gamma": {"distribution": "log_uniform_values", "min": 0.7, "max": 0.99},
                "scheduler_patience": {"values": [3, 5, 7]},
                "num_workers": {"value": 16},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 0},
                "start_season": {"value": 0},
                "num_features": {"distribution": "int_uniform", "min": 10, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model (Loss: BCE with Entropy)

In [6]:
create_submission(season=2025, artifact_name="0qeq5pdg:v5", submission_affix="weighted_avg_bce_entropy")

Config: {'config': "NeuralNetworkHyperparamConfig(hidden_layers=[128, 64], activation='elu', dropout=0.20283324507315925, batch_norm=True, input_dropout=0.0, learning_rate=0.00010906161385578576, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.0852016203217489, momentum=0.9, scheduler='exponential', scheduler_step_size=15, scheduler_gamma=0.7718575062077663, scheduler_patience=3, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='bce_entropy', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)", 'input_dim': 85, 'run_config': {'data_loader': 'weighted_season_average', 'num_features': 99, 'start_season': 0, 'valid_season': 0, 'data_loader_config': {'regular_weight': 0.7553080179979803, 'tourney_weight': 0.5651161623405925, 'discount_factor': 0.9273124186790386}}, 'neural_network_config': {'seed': 42, 'dropout': 0.20283324507315925, 'optimizer': 'ad

Seed set to 42
wandb:   1 of 1 files downloaded.  


Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_neural_network_weighted_avg_bce_entropy_2025.csv
